In [1]:
%%capture
# We're installing the latest Torch, Triton, OpenAI's Triton kernels, Transformers and Unsloth!
!pip install --upgrade -qqq uv
try: import numpy; get_numpy = f"numpy=={numpy.__version__}"
except: get_numpy = "numpy"
!uv pip install -qqq \
    "torch>=2.8.0" "triton>=3.4.0" {get_numpy} torchvision bitsandbytes "transformers>=4.55.3" \
    "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
    "unsloth[base] @ git+https://github.com/unslothai/unsloth" \
    git+https://github.com/triton-lang/triton.git@05b2c186c1b6c9a08375389d5efe9cb4c401c075#subdirectory=python/triton_kernels
!uv pip install transformers==4.55.4 pymongo vllm>=0.8.5
!uv pip install google-cloud-storage google-cloud-bigquery PyMuPDF tqdm 
!uv pip install titans-pytorch gradio gradio[mcp] db-dtypes google-cloud-bigquery-storage

In [2]:
!uv pip install protobuf==3.20.3

Using Python 3.11.13 environment at: /usr
Resolved 1 package in 6ms                                            
Uninstalled 1 package in 1ms
Installed 1 package in 2.89s                                
 - protobuf==6.32.1
 + protobuf==3.20.3


In [ ]:
# ===============================================================
# Cell 2 (Corrected): Load Data from BigQuery and Generate Embeddings
# ===============================================================
from unsloth import FastLanguageModel
from google.cloud import bigquery
from google.oauth2 import service_account # <-- ADD THIS IMPORT
from sentence_transformers import SentenceTransformer
import pandas as pd
from tqdm import tqdm
import os

# --- CONFIGURATION ---
GCP_PROJECT_ID = "silver-455021"
BQ_DATASET_ID = "dipg_knowledge_base"
BQ_TABLE_ID = "research_chunks"
MONGO_URI = ""

# --- Authentication Setup (Explicit Method) ---
# Define the path to your service account key file
SERVICE_ACCOUNT_FILE = ""

# Explicitly create credentials from the service account file
credentials = service_account.Credentials.from_service_account_file(SERVICE_ACCOUNT_FILE)
# ---------------------------------------------


# --- 1. Load Data from BigQuery ---
print("Querying BigQuery to get text chunks...")

# Pass the credentials directly to the client constructor
bq_client = bigquery.Client(credentials=credentials, project=GCP_PROJECT_ID)

query = f"""
    SELECT chunk_id, chunk_text
    FROM `{GCP_PROJECT_ID}.{BQ_DATASET_ID}.{BQ_TABLE_ID}`
"""
query_job = bq_client.query(query)
dataset_df = query_job.to_dataframe()
print(f"Successfully loaded {len(dataset_df)} chunks from BigQuery.")


# --- 2. Generate Embeddings ---
# (The rest of your code remains the same)
print("Loading embedding model...")
embedding_model = SentenceTransformer("unsloth/embeddinggemma-300m")

def get_embedding(text: str) -> list[float]:
    if not text or not isinstance(text, str) or not text.strip():
        return []
    embedding = embedding_model.encode(text)
    return embedding.tolist()

print("Generating embeddings for all text chunks...")
tqdm.pandas(desc="Embedding prompts")
dataset_df["embedding"] = dataset_df["chunk_text"].progress_apply(get_embedding)
print("\n✅ Embeddings generated successfully!")


# --- 3. Store Embeddings in MongoDB ---
import pymongo

print("Connecting to MongoDB Atlas...")
mongo_client = pymongo.MongoClient(MONGO_URI)
DB_NAME = "dipg_rag_db"
COLLECTION_NAME = "dipg_vectors"
db = mongo_client[DB_NAME]
collection = db[COLLECTION_NAME]

dataset_df_clean = dataset_df[dataset_df['embedding'].apply(lambda x: len(x) > 0)].copy()
documents_to_insert = dataset_df_clean[['chunk_id', 'embedding']].to_dict("records")

print("Ingesting vector data into MongoDB Atlas...")
collection.delete_many({})
collection.insert_many(documents_to_insert)
print("✅ Vector ingestion complete!")
print("IMPORTANT: Create a Vector Search Index in MongoDB on the 'embedding' field now.")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


2025-09-25 13:18:28.752395: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1758806309.083379      36 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1758806309.175549      36 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


INFO 09-25 13:18:59 [__init__.py:216] Automatically detected platform cuda.
ERROR 09-25 13:19:00 [fa_utils.py:57] Cannot use FA version 2 is not supported due to FA2 is only supported on devices with compute capability >= 8
🦥 Unsloth Zoo will now patch everything to make training faster!
Querying BigQuery to get text chunks...
Successfully loaded 76 chunks from BigQuery.
Loading embedding model...


modules.json:   0%|          | 0.00/573 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/997 [00:00<?, ?B/s]

...

Embedding prompts:  99%|█████████▊| 75/76 [00:17<00:00, 11.01it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embedding prompts: 100%|██████████| 76/76 [00:18<00:00,  4.21it/s]



✅ Embeddings generated successfully!
Connecting to MongoDB Atlas...
Ingesting vector data into MongoDB Atlas...
✅ Vector ingestion complete!
IMPORTANT: Create a Vector Search Index in MongoDB on the 'embedding' field now.
